# CDR-MLC v2: Congestion-Weighted Mixture of Experts

This notebook upgrades the leakage-safe CDR-MLC architecture while preserving its core:

- the same causal window of SynAck, AckDat, and TcpRtt;
- the same five statistics and three K-Means congestion regimes;
- three multiclass Random-Forest experts.

Hard partitioning is replaced by continuous congestion membership. Every training sample contributes to every expert with a different sample weight derived only from its distance to the three congestion centroids. At inference, expert class probabilities are combined with the corresponding congestion memberships.

No test label is used for clustering, membership, expert training, weighting, or prediction.


In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

MAIN = Path("CDR-MLC.ipynb")
if not MAIN.exists():
    MAIN = Path("CDR_MLC") / "CDR-MLC.ipynb"
namespace = {}
with MAIN.open(encoding="utf-8") as handle:
    main_notebook = json.load(handle)
exec(compile("".join(main_notebook["cells"][0]["source"]), str(MAIN), "exec"), namespace)
run_pipeline_from_two_files = namespace["run_pipeline_from_two_files"]
compute_sliding_window_stats = namespace["compute_sliding_window_stats"]

try:
    display
except NameError:
    display = print


def _memberships(distances, power=2.0):
    inverse = 1.0 / np.maximum(distances, 1e-9) ** power
    return inverse / inverse.sum(axis=1, keepdims=True)


def _aligned_probabilities(classifier, X, number_of_classes):
    raw = classifier.predict_proba(X)
    aligned = np.zeros((len(X), number_of_classes), dtype=np.float64)
    for column, label in enumerate(classifier.classes_):
        aligned[:, int(label)] = raw[:, column]
    return aligned


def run_cdr_mlc_v2(train_file, test_file, membership_power=2.0):
    # The existing hard model is retained as the direct baseline.
    baseline = run_pipeline_from_two_files(
        train_file=str(train_file), test_file=str(test_file),
        n_clusters=3, window_size=3,
        clustering_stats=["mean", "median", "std", "min", "max"],
    )
    train = baseline["train_df"]
    test = baseline["test_df"]
    features = baseline["classification_features"]
    target = baseline["target_column"]

    # Recreate the same label-free 15-dimensional congestion vectors.
    train_stats, _ = compute_sliding_window_stats(
        train[["SynAck", "AckDat", "TcpRtt"]],
        ["SynAck", "AckDat", "TcpRtt"], 3,
        ["mean", "median", "std", "min", "max"],
    )
    test_stats, _ = compute_sliding_window_stats(
        test[["SynAck", "AckDat", "TcpRtt"]],
        ["SynAck", "AckDat", "TcpRtt"], 3,
        ["mean", "median", "std", "min", "max"],
    )
    train_distances = baseline["kmeans_model"].transform(
        baseline["scaler"].transform(train_stats))
    test_distances = baseline["kmeans_model"].transform(
        baseline["scaler"].transform(test_stats))
    train_membership = _memberships(train_distances, membership_power)
    test_membership = _memberships(test_distances, membership_power)

    X_train = train[features].to_numpy()
    y_train = train[target].to_numpy()
    X_test = test[features].to_numpy()
    number_of_classes = len(baseline["label_encoder"].classes_)

    experts = []
    for expert_id in range(3):
        expert = RandomForestClassifier(
            n_estimators=80, random_state=42,
            class_weight="balanced", n_jobs=-1,
        )
        expert.fit(
            X_train, y_train,
            sample_weight=train_membership[:, expert_id],
        )
        experts.append(expert)

    probabilities = np.stack([
        _aligned_probabilities(expert, X_test, number_of_classes)
        for expert in experts
    ], axis=1)
    predictions = (
        probabilities * test_membership[:, :, None]
    ).sum(axis=1).argmax(axis=1)

    # Evaluation boundary: test labels first become visible here.
    y_test = test[target].to_numpy()
    v2_metrics = {
        "accuracy": accuracy_score(y_test, predictions),
        "precision_weighted": precision_score(
            y_test, predictions, average="weighted", zero_division=0),
        "recall_weighted": recall_score(
            y_test, predictions, average="weighted"),
        "f1_weighted": f1_score(
            y_test, predictions, average="weighted"),
        "f1_macro": f1_score(y_test, predictions, average="macro"),
    }
    metric_names = list(v2_metrics)
    table = pd.DataFrame([
        {"method": "hard_cdr_mlc", **{
            metric: baseline["test_results"][metric]
            for metric in metric_names}},
        {"method": "weighted_moe_v2", **v2_metrics},
    ])
    for metric in metric_names:
        base_value = table.loc[
            table["method"] == "hard_cdr_mlc", metric].iloc[0]
        table[f"{metric}_gain_pp"] = 100 * (table[metric] - base_value)

    display(table.round(4))
    return {
        "baseline": baseline,
        "experts": experts,
        "train_membership": train_membership,
        "test_membership": test_membership,
        "predictions": predictions,
        "comparison": table,
        "membership_power": membership_power,
    }


In [ ]:
# scenario_1: run independently
scenario_1_v2 = run_cdr_mlc_v2(
    "DATASETS/CDR-MLC/scale_1/Short/level_1.csv",
    "DATASETS/CDR-MLC/scale_1/Short/level_2.csv",
)


In [ ]:
# scenario_2: run independently
scenario_2_v2 = run_cdr_mlc_v2(
    "DATASETS/CDR-MLC/scale_1/Short/level_1.csv",
    "DATASETS/CDR-MLC/scale_1/Short/level_3.csv",
)


In [ ]:
# scenario_3: run independently
scenario_3_v2 = run_cdr_mlc_v2(
    "DATASETS/CDR-MLC/scale_1/Short/level_2.csv",
    "DATASETS/CDR-MLC/scale_1/Short/level_3.csv",
)


In [ ]:
# scenario_4: run independently
scenario_4_v2 = run_cdr_mlc_v2(
    "DATASETS/CDR-MLC/scale_1/Short/CDR-MLC-Shuffle.csv",
    "DATASETS/CDR-MLC/scale_1/Long/CDR-MLC-Shuffle.csv",
)


In [ ]:
# scenario_5: run independently
scenario_5_v2 = run_cdr_mlc_v2(
    "DATASETS/CDR-MLC/scale_1/Long/CDR-MLC-Shuffle.csv",
    "DATASETS/CDR-MLC/scale_1/Short/CDR-MLC-Shuffle.csv",
)
